
# 🌍 Spanish → English with **Meta-Llama-3-8B-Instruct**
This notebook loads **meta-llama/Meta-Llama-3-8B-Instruct** locally with [Transformers] and lets you:
- Translate individual snippets of Spanish to natural English.
- Translate entire `.srt` subtitle files while preserving timing.

> **Notes**
> - For best performance, a CUDA GPU is recommended. With ~6 GB VRAM you should use **4‑bit** loading (`bitsandbytes`). CPU also works, but it's slower.
> - You’ll be prompted for your **Hugging Face token** at runtime (not stored in the notebook).  
> - If PyTorch isn't installed yet, follow the **PyTorch install** cell first.



## 1) (Optional) Install PyTorch
If you don't have PyTorch yet, install it **matching your system**. See the official selector: https://pytorch.org/get-started/locally/

On Windows with CUDA 12.4 toolkit, for example:
```powershell
# Example (adapt to your CUDA / Python):
pip install --index-url https://download.pytorch.org/whl/cu124 torch torchvision torchaudio
```
If you only have CPU:
```bash
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
```


In [2]:

# 2) Install/upgrade the required Python libraries.
# - bitsandbytes enables 4-bit/8-bit quantized loading (saves VRAM).
# - sentencepiece is required for Llama tokenization.
# - accelerate helps with device placement and offloading.
# If any installs fail in your environment, re-run this cell after adjusting your CUDA/PyTorch setup.
%pip install -q -U transformers accelerate huggingface_hub sentencepiece bitsandbytes


Note: you may need to restart the kernel to use updated packages.



## 3) Log in to Hugging Face
You'll be asked for your HF token (it won't be saved in the notebook).  
If your account hasn't accepted the Meta Llama 3 license terms yet, do that on the model page first.



## 4) Load **Meta-Llama-3-8B-Instruct**
This cell tries to load the model using the **best available** config:
- **If CUDA + bitsandbytes** → 4‑bit quantization (`load_in_4bit=True`) with automatic device placement.
- Else **if CUDA** → fp16 on GPU.
- Else → CPU (slower).
You can override defaults in the block below.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
#https://huggingface.co/bigscience/bloomz-7b1-mt MEJOR MODELO

bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.pad_token_id = tok.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_4bit,
    device_map={"": 0},          # todo en GPU (6 GB lo aguanta bien)
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)

# ids útiles para stop
EOS_ID = tok.eos_token_id
IM_END_ID = tok.convert_tokens_to_ids("<|im_end|>")
STOP_IDS = [i for i in [EOS_ID, IM_END_ID] if i is not None]


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


## 5) Translation helper
The function below uses Llama 3's chat template to prompt translation.  
It returns **only the English translation** (no extra explanations).


In [4]:
from transformers import StoppingCriteria, StoppingCriteriaList
from transformers.utils import logging
logging.set_verbosity_error()
# o:
import os; os.environ["TRANSFORMERS_VERBOSITY"] = "error"

class StopOnAnyId(StoppingCriteria):
    def __init__(self, stop_ids):
        self.stop_ids = set(stop_ids)
    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[0, -1].item() in self.stop_ids

def translate_es_to_en(text: str, max_new_tokens: int = 128) -> str:
    system = (
        "You are a professional translator. Translate Spanish to natural, fluent English. "
        "Use idiomatic meaning (not literal). Preserve names and numbers. Output ONLY the translation."
    )
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": text}
    ]

    # prompt del modelo (chat template Qwen)
    input_ids = tok.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    gen_ids = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,                 # greedy determinista
        use_cache=False,                 # ahorra VRAM
        eos_token_id=STOP_IDS,           # parar en <|im_end|> y/o eos
        pad_token_id=tok.pad_token_id,
        no_repeat_ngram_size=3,
        repetition_penalty=1.1,
        stopping_criteria=StoppingCriteriaList([StopOnAnyId(STOP_IDS)]),
        return_dict_in_generate=False,
    )

    new_tokens = gen_ids[0, input_ids.shape[1]:]
    out = tok.decode(new_tokens, skip_special_tokens=True).strip()

    # Limpieza defensiva: si se coló un prefijo o líneas extra, coge la primera “línea útil”
    out = out.split("\n")[0].strip().strip('"')
    return out



## 6) Quick demo
Try a short sentence in Spanish.


In [5]:

sample = "Pero, tía, me echaba unos polvos."
print("Spanish:", sample)
print("English:", translate_es_to_en(sample))


Spanish: Pero, tía, me echaba unos polvos.
English: But Auntie, I was shagging around.



## 7) Translate an `.srt` file (Spanish → English)
This preserves all timing, only replacing the subtitle text.


In [ ]:
# ==== Traducir SRT ES->EN imprimiendo progreso línea a línea ====
import re, io, os, sys, torch

PRINT_LINES = True   # pon a False si no quieres ver cada línea

def parse_srt(srt_text: str):
    srt_text = srt_text.replace("\r\n", "\n").replace("\r", "\n").strip()
    blocks = re.split(r'\n{2,}', srt_text)
    entries = []
    for block in blocks:
        lines = block.strip().split("\n")
        if len(lines) >= 3:
            entries.append({
                "index": lines[0].strip(),
                "times": lines[1].strip(),
                "text":  "\n".join(lines[2:]).strip()
            })
    return entries

def write_srt(entries):
    out = []
    for i, e in enumerate(entries, start=1):
        out += [str(i), e["times"], e["text"], ""]
    return "\n".join(out).rstrip() + "\n"

def _print_line(i, n, j, m, es, en):
    es_short = es if len(es) <= 80 else es[:77] + "..."
    en_short = en if len(en) <= 80 else en[:77] + "..."
    print(f"[{i}/{n}] línea {j}/{m}\n  ES: {es_short}\n  EN: {en_short}\n", flush=True)

# ---- Rutas ----
srt_in  = "transcripcion_vad_silero.srt"   # <-- tu SRT en español
srt_out = "output_en.srt"                  # <-- salida en inglés

if os.path.exists(srt_in):
    with io.open(srt_in, "r", encoding="utf-8") as f:
        srt_text = f.read()

    entries = parse_srt(srt_text)
    total_blocks = len(entries)
    print(f"Found {total_blocks} subtitle blocks. Translating...\n")

    model.eval()
    new_entries = []
    with torch.no_grad():
        for i, e in enumerate(entries, start=1):
            src_block = e["text"]
            lines = (src_block.split("\n") if src_block else [""])
            translated_lines = []
            for j, line in enumerate(lines, start=1):
                stripped = line.strip()
                if stripped:
                    try:
                        en = translate_es_to_en(stripped, max_new_tokens=80)
                    except Exception as ex:
                        en = stripped  # fallback para no romper el SRT
                    translated_lines.append(en)
                    if PRINT_LINES:
                        _print_line(i, total_blocks, j, len(lines), stripped, en)
                else:
                    translated_lines.append("")
                    if PRINT_LINES:
                        _print_line(i, total_blocks, j, len(lines), "", "")
            new_entries.append({"times": e["times"], "text": "\n".join(translated_lines)})

            # Marca de progreso por bloque
            print(f"  ✓ bloque {i}/{total_blocks}\n", flush=True)

    out_text = write_srt(new_entries)

Found 296 subtitle blocks. Translating...

[1/296] línea 1/1
  ES: ¡Gracias!
  EN: Thanks!

  ✓ bloque 1/296

[2/296] línea 1/1
  ES: Me viene Pilar que a ver si le puedo llevar en su coche al banco.
  EN: Pilar's coming; I could give her a ride to the bank in my car.

  ✓ bloque 2/296

[3/296] línea 1/1
  ES: Una faena. Me habían quitado todos los puntos.
  EN: One thing. They took all my points away.

  ✓ bloque 3/296

[4/296] línea 1/1
  ES: Nada, no se tarda nada.
  EN: Nothing, it doesn't take long at all.

  ✓ bloque 4/296

[5/296] línea 1/1
  ES: Un momentito.
  EN: A moment, if I may.

  ✓ bloque 5/296

[6/296] línea 1/1
  ES: Un momentito para esta...
  EN: A minute, if that...

  ✓ bloque 6/296



In [ ]:
# ==== Guardar SRT en inglés y traducir EN->XX desde `new_entries` (sin zh-hant) ====
import os, io, re, torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from IPython.display import FileLink, display

EN_TO_MODEL = {
    "de": "Helsinki-NLP/opus-mt-en-de",       # Alemán
    "fr": "Helsinki-NLP/opus-mt-en-fr",       # Francés
    "zh-hans": "Helsinki-NLP/opus-mt-en-zh",  # Chino simplificado
    "ar": "Helsinki-NLP/opus-mt-en-ar",       # Árabe
    "ru": "Helsinki-NLP/opus-mt-en-ru",       # Ruso
    "ja": "Helsinki-NLP/opus-mt-en-ja"        # Japonés
    # "ko": usar M2M100 si te interesa (facebook/m2m100_418M)
}

target_lang = "de"                 # "en","de","fr","zh-hans","ar","ru","ja"
srt_en_out = "output_en.srt"
srt_xx_out = f"output_{target_lang}.srt"
PRINT_LINES = True
BATCH_SIZE = 16

def write_srt(entries):
    out = []
    for i, e in enumerate(entries, start=1):
        out += [str(i), e["times"], e["text"], ""]
    return "\n".join(out).rstrip() + "\n"

def _print_line(i, n, j, m, src, tgt, tag="EN"):
    if not PRINT_LINES: return
    s = src if len(src) <= 80 else src[:77] + "..."
    t = tgt if len(tgt) <= 80 else tgt[:77] + "..."
    print(f"[{i}/{n}] línea {j}/{m}\n  EN: {s}\n  {tag}: {t}\n", flush=True)

def is_meta_tag(line: str) -> bool:
    return bool(re.fullmatch(r"\[[^\]]+\]", line.strip()))

if "new_entries" not in globals() or not isinstance(new_entries, list) or not new_entries:
    raise RuntimeError("❌ `new_entries` no está definido o está vacío. Ejecuta antes la celda ES->EN que lo crea.")

# Guardar EN solo si se pide inglés
if target_lang == "en":
    en_text = write_srt(new_entries)
    with io.open(srt_en_out, "w", encoding="utf-8") as f:
        f.write(en_text)
    print(f"✅ Guardado SRT en inglés: {srt_en_out}")
    display(FileLink(srt_en_out))
else:
    if target_lang not in EN_TO_MODEL:
        raise ValueError(f"Idioma '{target_lang}' no soportado. Usa uno de: en, " + ", ".join(EN_TO_MODEL.keys()))

    model_name = EN_TO_MODEL[target_lang]
    device = 0 if torch.cuda.is_available() else -1
    tok_mt = AutoTokenizer.from_pretrained(model_name)
    mdl_mt = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    translator = pipeline("translation", model=mdl_mt, tokenizer=tok_mt, device=device)

    print(f"Traduciendo EN → {target_lang}...\n")
    xx_entries = []
    total = len(new_entries)

    for i, e in enumerate(new_entries, start=1):
        src_lines = e["text"].split("\n") if e["text"] else [""]
        tgt_lines = [""] * len(src_lines)

        # Lote solo con líneas traducibles
        idxs, batch = [], []
        for j, line in enumerate(src_lines):
            s = line.strip()
            if s and not is_meta_tag(s):
                idxs.append(j); batch.append(s)

        if batch:
            outs = translator(batch, max_length=512, batch_size=BATCH_SIZE)
            outs_text = [o["translation_text"] for o in outs]
            for k, j in enumerate(idxs):
                tgt_lines[j] = outs_text[k]
                _print_line(i, total, j+1, len(src_lines), src_lines[j], tgt_lines[j], tag=target_lang.upper())

        # Copia tal cual vacías/etiquetas
        for j, line in enumerate(src_lines):
            if line.strip() == "" or is_meta_tag(line):
                tgt_lines[j] = line

        xx_entries.append({"times": e["times"], "text": "\n".join(tgt_lines)})
        if i % 20 == 0 or i == total:
            print(f"  ✓ bloque {i}/{total}\n", flush=True)

    xx_text = write_srt(xx_entries)
    with io.open(srt_xx_out, "w", encoding="utf-8") as f:
        f.write(xx_text)
    print(f"✅ Guardado SRT {target_lang}: {srt_xx_out}")
    display(FileLink(srt_xx_out))


Traduciendo EN → de...

[1/296] línea 1/1
  EN: Thanks!
  DE: Vielen Dank!

[2/296] línea 1/1
  EN: Pilar's coming; I could give her a ride to the bank in my car.
  DE: Pilar kommt, ich könnte sie in meinem Auto zur Bank fahren.

[3/296] línea 1/1
  EN: One thing. They took all my points away.
  DE: Sie haben mir alle meine Punkte weggenommen.

[4/296] línea 1/1
  EN: Nothing, it doesn't take long at all.
  DE: Nichts, es dauert nicht lange.

[5/296] línea 1/1
  EN: A moment, if I may.
  DE: Einen Moment, wenn ich darf.

[6/296] línea 1/1
  EN: A minute, if that...
  DE: Eine Minute, wenn das...

[7/296] línea 1/1
  EN: It's going to Cobo calleja and coming back.
  DE: Es geht nach Cobo calleja und kommt zurück.

[8/296] línea 1/1
  EN: They told me they gave loans without thinking.
  DE: Sie sagten mir, sie hätten Kredite gegeben, ohne nachzudenken.

[9/296] línea 1/1
  EN: I have no idea who would tell him that.
  DE: Ich habe keine Ahnung, wer ihm das sagen würde.

[10/296] línea 1/

c:\Users\carlos.basallote\Desktop\TFM\salidas\output_de.srt


## 8) Troubleshooting
- **CUDA not available**: Install the right PyTorch build for your GPU/CUDA. Then restart the kernel.
- **OOM (out of memory)** on GPU: Ensure 4‑bit is enabled (default if VRAM ≤ 9 GB). Close other GPU apps.
- **bitsandbytes errors on Windows**: Update `bitsandbytes` to the latest version. If it still fails, fallback to fp16 (more VRAM) or CPU.
- **Slow translations**: Use shorter `max_new_tokens`, set `temperature=0.0`, and prefer GPU.
- **Token / 403 errors**: Make sure your HF account has access and you’re logged in.


In [ ]:
# ==== Selector de idioma y guardado de SRT (desde `new_entries`) ====
import os, io, re, torch
import ipywidgets as w
from IPython.display import display, clear_output, FileLink
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# Modelos EN->XX (puedes añadir/quitar pares aquí)
EN_TO_MODEL = {
    "de": "Helsinki-NLP/opus-mt-en-de",   # Alemán
    "fr": "Helsinki-NLP/opus-mt-en-fr",   # Francés
    "it": "Helsinki-NLP/opus-mt-en-it",   # Italiano
    "pt": "Helsinki-NLP/opus-mt-tc-big-en-pt",   # Portugués
    "nl": "Helsinki-NLP/opus-mt-en-nl",   # Neerlandés
    "sv": "Helsinki-NLP/opus-mt-en-sv",   # Sueco
    "da": "Helsinki-NLP/opus-mt-en-da",   # Danés
    "cs": "Helsinki-NLP/opus-mt-en-cs",   # Checo
    "pl": "gsarti/opus-mt-tc-en-pl",   # Polaco
    "uk": "Helsinki-NLP/opus-mt-en-uk",   # Ucraniano
    "el": "Helsinki-NLP/opus-mt-en-el",   # Griego
    "he": "Helsinki-NLP/opus-mt-en-he",   # Hebreo
    "tr": "Helsinki-NLP/opus-mt-en-tr",   # Turco
    "ro": "Helsinki-NLP/opus-mt-en-ro",   # Rumano
    "zh-hans": "Helsinki-NLP/opus-mt-en-zh",  # Chino simplificado
    "ar": "Helsinki-NLP/opus-mt-en-ar",   # Árabe
    "ru": "Helsinki-NLP/opus-mt-en-ru",   # Ruso
    "ja": "Helsinki-NLP/opus-mt-en-ja",   # Japonés
    "ko": "Helsinki-NLP/opus-mt-tc-big-en-ko" #Coreano
}

# Nombres bonitos para el desplegable
LANG_LABELS = {
    "en": "Inglés",
    "de": "Alemán",
    "fr": "Francés",
    "it": "Italiano",
    "pt": "Portugués",
    "nl": "Neerlandés",
    "sv": "Sueco",
    "da": "Danés",
    "cs": "Checo",
    "pl": "Polaco",
    "uk": "Ucraniano",
    "el": "Griego",
    "he": "Hebreo",
    "tr": "Turco",
    "ro": "Rumano",
    "zh-hans": "Chino (simplificado)",
    "ar": "Árabe",
    "ru": "Ruso",
    "ja": "Japonés",
    "ko": "Coreano",
}


def label(code: str) -> str: return LANG_LABELS.get(code, code)


# -------- Helpers SRT --------
def write_srt(entries):
    out=[]
    for i,e in enumerate(entries,1):
        out += [str(i), e["times"], e["text"], ""]
    return "\n".join(out).rstrip()+"\n"

def is_meta_tag(line: str) -> bool:
    return bool(re.fullmatch(r"\[[^\]]+\]", line.strip()))

def translate_srt_from_en(entries_en, target_lang, batch_size=16, print_lines=True):
    """entries_en: [{'times':..,'text':..}, ...] en INGLÉS"""
    if target_lang == "en":
        return entries_en  # sin cambios

    model_name = EN_TO_MODEL[target_lang]
    device = 0 if torch.cuda.is_available() else -1
    tok_mt = AutoTokenizer.from_pretrained(model_name)
    mdl_mt = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    translator = pipeline("translation", model=mdl_mt, tokenizer=tok_mt, device=device)

    total = len(entries_en)
    out_entries = []
    for i, e in enumerate(entries_en, start=1):
        src_lines = e["text"].split("\n") if e["text"] else [""]
        tgt_lines = [""] * len(src_lines)

        # Lote solo con líneas traducibles (no vacías ni [Música])
        idxs, batch = [], []
        for j, line in enumerate(src_lines):
            s = line.strip()
            if s and not is_meta_tag(s):
                idxs.append(j); batch.append(s)

        # Traducción por lotes
        if batch:
            outs = translator(batch, max_length=512, batch_size=batch_size)
            outs_text = [o["translation_text"] for o in outs]
            for k, j in enumerate(idxs):
                tgt_lines[j] = outs_text[k]
                if print_lines:
                    es = src_lines[j]
                    en = tgt_lines[j]
                    es_s = es if len(es)<=80 else es[:77]+"..."
                    en_s = en if len(en)<=80 else en[:77]+"..."
                    print(f"[{i}/{total}] línea {j+1}/{len(src_lines)}\n  EN: {es_s}\n  {target_lang.upper()}: {en_s}\n")

        # Copia vacías/etiquetas tal cual
        for j, line in enumerate(src_lines):
            if line.strip()=="" or is_meta_tag(line):
                tgt_lines[j] = line

        out_entries.append({"times": e["times"], "text": "\n".join(tgt_lines)})

        if i % 20 == 0 or i == total:
            print(f"  ✓ bloque {i}/{total}\n", flush=True)
    return out_entries

# -------- Widgets --------
if "new_entries" not in globals() or not isinstance(new_entries, list) or not new_entries:
    raise RuntimeError("❌ `new_entries` no está definido o está vacío. Ejecuta antes la celda ES->EN que lo crea.")

# "en" primero y luego los del diccionario (puedes ordenar alfabéticamente si prefieres)
lang_codes = ["en"] + list(EN_TO_MODEL.keys())
# Si quieres orden alfabético por nombre visible, usa:
# lang_codes = ["en"] + sorted(EN_TO_MODEL.keys(), key=lambda c: LANG_LABELS.get(c, c))

lang_options = [(label("en"), "en")] + [(label(k), k) for k in EN_TO_MODEL.keys()]

dd_lang = w.Dropdown(options=lang_options, value="de", description="Destino:")
cb_print = w.Checkbox(value=True, description="Mostrar líneas")
bs = w.IntSlider(value=16, min=4, max=64, step=4, description="Batch")
btn = w.Button(description="Traducir y guardar", button_style="primary")
out = w.Output()

def on_click(_):
    with out:
        clear_output(wait=True)
        target = dd_lang.value
        nombre = LANG_LABELS.get(target, target)
        print(f"Destino: {nombre}\n")
        if target == "en":
            srt_path = "output_en.srt"
            with io.open(srt_path, "w", encoding="utf-8") as f:
                f.write(write_srt(new_entries))
            print(f"✅ Guardado SRT {nombre}: {srt_path}")
            display(FileLink(srt_path))
            return

        if target not in EN_TO_MODEL:
            print(f"⚠️ Idioma '{target}' no soportado en EN_TO_MODEL.")
            return

        xx_entries = translate_srt_from_en(new_entries, target, batch_size=bs.value, print_lines=cb_print.value)
        srt_path = f"output_{target}.srt"
        with io.open(srt_path, "w", encoding="utf-8") as f:
            f.write(write_srt(xx_entries))
        print(f"✅ Guardado SRT {LANG_LABELS.get(target, target)}: {srt_path}")
        display(FileLink(srt_path))

btn.on_click(on_click)
display(w.HBox([dd_lang, cb_print, bs, btn]), out)


KeyboardInterrupt: 